<a href="https://colab.research.google.com/github/simb-01/CM2013-Signal-Processing-/blob/main/Recon_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image reconstruction lab


---

You are given simulated **parallel-beam CT sinograms** of an anatomical phantom,
at three noise levels. The goal is to reconstruct them into images by implementing
filtered backprojection yourself in [ODL](https://odl.readthedocs.io/), and then
to experiment using different reconstruction filters.


Cells marked **`# TODO`** are yours to complete.
as-is.

Useful ODL pages: [spaces](https://odl.readthedocs.io/getting_started/in_depth/space_concept.html),
[operators](https://odl.readthedocs.io/getting_started/in_depth/operator_concept.html),
[`RayTransform`](https://odl.readthedocs.io/generated/odl.applications.tomo.operators.RayTransform.html),
[`FourierTransform`](https://odl.readthedocs.io/generated/odl.trafos.FourierTransform.html).


### The data

All files live in `lab_data/`.

| File | What it is |
|---|---|
| `2d_phantom.npy` | ground-truth attenuation $\mu$, in $1/\mathrm{cm}$ |
| `2d_sinogram_clean.npy` | noise-free line integrals $p = \int_\ell \mu\,\mathrm{d}\ell$ |
| `2d_sinograms_std_0.05.npy`, `_0.1.npy`, `_0.5.npy` | the same with additive Gaussian noise |

The geometry that produced them. **All lengths are in centimetres**, so that
$\mu$ comes out in $1/\mathrm{cm}$:

- reconstruction volume $512\times 512$ pixels of $0.1\,\mathrm{cm}$, centred at the origin, i.e. spanning $[-25.6,\,25.6]\,\mathrm{cm}$ in both $x$ and $y$
- $180$ projections, uniformly spaced on $[0, \pi)$
- a 1D detector of $740$ bins of $0.1\,\mathrm{cm}$, spanning $[-37,\,37]\,\mathrm{cm}$ (the image diagonal, rounded up to a whole cm)
- parallel beam, 2D (one axial slice of the anatomical phantom)

The data was generated with exactly this geometry, so if you get it right your
own forward operator applied to `2d_phantom.npy` reproduces
`2d_sinogram_clean.npy` to the last bit. That is worth checking in section 2.


In [1]:
# Colab setup
!pip install odl==1.0.0b1

import os, urllib.request
BASE = "https://raw.githubusercontent.com/debienicolas/reconlab/main/"
FILES = ["2d_phantom.npy", "2d_sinogram_clean.npy",
         "2d_sinograms_std_0.05.npy", "2d_sinograms_std_0.1.npy",
         "2d_sinograms_std_0.5.npy"]
os.makedirs("lab_data", exist_ok=True)
for f in FILES:
    if not os.path.exists(f"lab_data/{f}"):
        urllib.request.urlretrieve(BASE + f, f"lab_data/{f}")
print(sorted(os.listdir("lab_data")))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.1/796.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 52.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


['2d_phantom.npy', '2d_sinogram_clean.npy', '2d_sinograms_std_0.05.npy', '2d_sinograms_std_0.1.npy', '2d_sinograms_std_0.5.npy']


In [ ]:
import odl
import numpy as np
import matplotlib.pyplot as plt

### 1. Set up the geometry of the problem

Define an odl `Parallel2dGeometry` that matches the
acquisition description above.


In [ ]:
# TODO

### 2. Ray transform and the sinogram

Build the forward operator on the geometry you just defined, load a sinogram from file and inspect it.

In [ ]:
# TODO


### 3. Backprojection

Implement unfiltered backprojection from scratch and compare it to
`ray_trafo.adjoint`. They should agree. (ODL's adjoint already includes the
angular weight $\mathrm{d}\theta$; yours must too.)

$$
\mathcal{R}^{*}p\,(x_0, y_0) \;=\; \int_{\theta=0}^{\pi}
p\bigl(\theta,\;x_0\cos\theta+y_0\sin\theta\bigr)\,\mathrm{d}\theta
$$

(no $1/\pi$ out front: that is the convention `ray_trafo.adjoint` uses, and it
is what makes the ramp filter in section 4 the exact inverse.)

useful methods: np.meshgrid()


In [ ]:
def my_backproject(sino, geom, recon_space):
    p = sino.asarray()
    # TODO:
    #   - pixel centres from recon_space.grid.coord_vectors
    #   - detector bin centres from geom.det_partition.grid
    #   - angles from geom.angles, dtheta from geom.motion_partition.cell_sides
    #   - for each angle, t = x * cos θ + y * sin θ  (geom.det_axis(theta))
    #   - interpolate the projection onto every pixel (np.interp) and accumulate
    #   - multiply the result by dtheta



# TODO plot your backprojection implementation
# TODO compare it to the adjoint of the forward operator


### 4. The ramp filter: the analytical inverse


#### Fourier slice theorem

The 1D Fourier transform of the projection at angle $\theta$ equals a radial
slice, at that same angle, through the 2D Fourier transform of the object:

$$\mathcal{F}_1\{p_\theta\}(\omega) \;=\; \mathcal{F}_2\{f\}(\omega\cos\theta,\ \omega\sin\theta)$$

So a complete scan tells you $\mathcal{F}_2\{f\}$ on a **polar** grid: one radial
line through the origin per projection angle.

#### From polar frequencies back to the image

To recover $f$ you apply the inverse 2D Fourier transform, but that is an
integral in *Cartesian* frequency coordinates $(u,v)$, and your data lives in
polar coordinates $(\omega, \theta)$. Changing variables carries a Jacobian:

$$\mathrm{d}u\,\mathrm{d}v \;=\; |\omega|\ \mathrm{d}\omega\,\mathrm{d}\theta$$

Substituting it in gives the filtered backprojection formula:

$$f(x,y) \;=\; \int_0^{\pi} \left[\int_{-\infty}^{\infty} \mathcal{F}_1\{p_\theta\}(\omega)\ |\omega|\ e^{2\pi i \omega t}\,\mathrm{d}\omega\right] \mathrm{d}\theta,
\qquad t = x\cos\theta + y\sin\theta$$


$|\omega|$ is a **Jacobian**. It falls out of a change of variables,
and together with the backprojection it is the *exact analytical inverse* of the
ray transform.


In [ ]:
# helper method to display multiple images
def show(*images, titles=None, vmin=0.0, vmax=0.30):
    """Side by side on a fixed grey window in 1/cm, in chest-CT orientation."""
    fig, axes = plt.subplots(1, len(images), figsize=(4.6 * len(images), 4.6))
    axes = np.atleast_1d(axes)
    for ax, im in zip(axes, images):
        arr = im.asarray() if hasattr(im, "asarray") else np.asarray(im)
        h = ax.imshow(arr.T, vmin=vmin, vmax=vmax, cmap="gray")
        ax.axis("off")
    if titles:
        for ax, t in zip(axes, titles):
            ax.set_title(t, fontsize=10)
    plt.colorbar(h, ax=axes.tolist(), fraction=0.03)
    plt.show()

#### Writing it in ODL

Filterbackprojection is a composition of multiple operations, try to formulate the operations present in FBP.
Once you have done this, define the operators using ODL and chain the operators.

Here are some relevant ODL operators:
- `odl.trafos.FourierTransform`
- `odl.ResizingOperator`
- chaining operators is done using `*`


- ODL's frequency axis is *angular*, in $\mathrm{rad}/\mathrm{cm}$. The $|\omega|$ of the
  formula above is in cycles/cm, so divide the coordinate by $2\pi$ before taking
  the absolute value. Skip this and your $\mu$ is off by a factor of $2\pi$.



In [ ]:
# TODO: Define your fbp ODL operator


Once you have your own FBP operator, compare the reconstruction against the ground truth phantom.

In [ ]:
# TODO: Compare against the ground truth


# TODO: plot several profiles of the reconstructions and the ground truth



In [ ]:
# TODO: plot a single sinogram projection and the corresponding profile after applying the ramp filter
# Why does it have negative values?

In [ ]:
# TODO: reconstruct the noisy sinograms using your FBP operator. Why is the ramp filter problematic when reconstructing

### 5. Low-pass filters

Applying FBP on noisy data fails, so we can add an additional low-pass filter $W(\omega)$:

$$|\omega| \;\longrightarrow\; |\omega|\, W(\omega)$$

$W$ is *chosen*, not derived.

The classical windows, written in the normalised frequency
$w = |\omega|/(\omega_{\mathrm{cut}}\,\omega_{\mathrm{Nyquist}})$ and set to
zero for $w > 1$:

| name | $W(w)$ |
|---|---|
| Ram-Lak | $1$ |
| Shepp–Logan | $\mathrm{sinc}(w/2)$ |
| cosine | $\cos(\pi w/2)$ |
| Hamming | $0.54 + 0.46\cos(\pi w)$ |
| Hann | $\tfrac12\bigl(1 + \cos(\pi w)\bigr)$ |

`cutoff` $\in (0, 1]$ is a second knob: the fraction of Nyquist at which the
window reaches zero.



In [ ]:
WINDOWS = ["ram-lak", "shepp-logan", "cosine", "hamming", "hann"]

# TODO: Calculate the Nyquist frequency
nyquist = None

def window_gain(omega, window="ram-lak", cutoff=1.0):
    """|omega| * W(omega)"""
    omega = np.abs(omega)
    w = omega / (cutoff * nyquist)
    # TODO: build `apod` for each name in WINDOWS from the table above.
    #       Then zero it where w > 1, and return omega * apod.
    raise NotImplementedError


def make_filter(window="ram-lak", cutoff=1.0):
    """The gain |omega| W(omega), as an element of the frequency space."""
    # TODO:
    raise NotImplementedError


def make_fbp(window="ram-lak", cutoff=1.0):
    """FBP operator: adjoint o F^{-1} o (|omega| W) o F."""
    # TODO: the same composition as my_fbp, with make_filter(...) in place of ramp.
    raise NotImplementedError



#TODO: plot the the various filters


#### Sweep over windows

Reconstruct the different noisy sinograms using different low-pass filters.


In [ ]:
# TODO: plot the results


#### Cutoff versus shape

The window *shape* is one knob. `cutoff` is another: it is the fraction of
Nyquist at which $W$ is forced to zero, so `cutoff = 0.5` throws away
everything above half the detector's own resolution, regardless of which
window you picked. Hold the window fixed and sweep the cutoff on a
single noisy sinogram.


In [ ]:
# TODO: tune the cutoff and plot the results
